# Phase 0 — VRAM spike

**One question:** does LatentSync 1.6 inference at 512 fit in Kaggle's 16 GB?

The whole 1080p plan rests on this. LatentSync 1.6 was retrained on 512×512
video to fix 1.5's blurry teeth and lips, and that larger face crop is what
makes a 1080p render worthwhile. If it does not fit, the fallback is 1.5 at 256
with a 720p output — which is a perfectly good plan, just a different one.

The widely-quoted "18 GB" and "30 GB" figures for LatentSync are **training**
requirements. Inference is far lighter. But "far lighter" is not a number, so
this notebook measures it.

## Before you run

1. **Settings → Accelerator → GPU** (P100 or T4 ×2). Without this the notebook
   exits immediately.
2. **Settings → Internet → On** (needed to clone the repo and pull weights).
3. Attach your private `talkinghead-assets` dataset via **+ Add Input** if you
   have one. Optional — the notebook falls back to a public sample clip.

Runtime is roughly 15–25 minutes, most of it downloading weights. It draws
against your 30 hr/week GPU quota, so this costs about 0.4 hours of it.

**Read the verdict in the last cell.** It tells you which profile to use.

## 1. Confirm the GPU

Fail fast: a CPU session would spend twenty minutes downloading weights before
discovering it cannot answer the question.

In [ ]:
import subprocess, sys

try:
    import torch
except ImportError:
    sys.exit("torch is not available in this session.")

if not torch.cuda.is_available():
    raise SystemExit(
        "No CUDA GPU in this session.\n"
        "Settings -> Accelerator -> GPU (P100 or T4 x2), then re-run."
    )

props = torch.cuda.get_device_properties(0)
TOTAL_VRAM_GB = props.total_memory / 1024**3

print(f"GPU            {props.name}")
print(f"total VRAM     {TOTAL_VRAM_GB:.2f} GB")
print(f"torch          {torch.__version__}")
print(f"CUDA           {torch.version.cuda}")
print(f"visible GPUs   {torch.cuda.device_count()}")
print()
print(subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total,memory.used", "--format=csv"],
    capture_output=True, text=True,
).stdout)

## 2. Clone LatentSync from the first-party repo

`github.com/bytedance/LatentSync` only. Apache 2.0 on code *and* weights, which
is why this project can use it for client work at all. No mirrors, no ComfyUI
wrappers, no re-uploads.

In [ ]:
import os
from pathlib import Path

WORK = Path("/kaggle/working")
REPO_DIR = WORK / "LatentSync"

if not REPO_DIR.exists():
    !git clone --depth 1 https://github.com/bytedance/LatentSync.git {REPO_DIR}

os.chdir(REPO_DIR)
!git log -1 --format="pinned commit: %H%n            %s"

# Record the SHA -- once this spike passes, freeze it in config.TRUSTED_SOURCES
# so a future upstream change cannot silently alter behaviour.
LATENTSYNC_SHA = subprocess.run(
    ["git", "rev-parse", "HEAD"], capture_output=True, text=True, cwd=REPO_DIR
).stdout.strip()
print(f"\nSHA to pin: {LATENTSYNC_SHA}")

## 3. Find the 512 config

Per ByteDance's 1.6 changelog, switching from 1.5 to 1.6 means loading the 1.6
checkpoint and changing the resolution in the U-Net config — the architecture is
unchanged.

Rather than hard-coding a filename that may have moved, this lists what the repo
actually ships and picks by resolution. If the detection is wrong you will see it
here rather than in a confusing failure later.

In [ ]:
import yaml

configs = sorted((REPO_DIR / "configs").rglob("*.yaml"))
print("available configs:")
for c in configs:
    print(f"  {c.relative_to(REPO_DIR)}")

def declared_resolution(path):
    """Pull the resolution out of a U-Net config, wherever it is nested."""
    try:
        data = yaml.safe_load(path.read_text())
    except Exception:
        return None
    stack = [data]
    while stack:
        node = stack.pop()
        if isinstance(node, dict):
            for key, value in node.items():
                if key == "resolution" and isinstance(value, int):
                    return value
                stack.append(value)
        elif isinstance(node, list):
            stack.extend(node)
    return None

print("\nresolutions found:")
unet_configs = {}
for c in configs:
    res = declared_resolution(c)
    if res:
        unet_configs[c] = res
        print(f"  {res:4d}  {c.relative_to(REPO_DIR)}")

# Prefer a config that already declares 512; otherwise take the highest and
# override it below.
if not unet_configs:
    raise SystemExit("No U-Net config with a resolution field found -- inspect the listing above.")

UNET_CONFIG = max(unet_configs, key=lambda c: unet_configs[c])
print(f"\nusing: {UNET_CONFIG.relative_to(REPO_DIR)} (resolution {unet_configs[UNET_CONFIG]})")

In [ ]:
TARGET_RESOLUTION = 512

# Force 512 regardless of what the chosen config shipped with -- that is the
# whole point of the measurement.
cfg = yaml.safe_load(UNET_CONFIG.read_text())

def set_resolution(node, value):
    changed = False
    if isinstance(node, dict):
        for key in list(node):
            if key == "resolution" and isinstance(node[key], int):
                node[key] = value
                changed = True
            else:
                changed |= set_resolution(node[key], value)
    elif isinstance(node, list):
        for item in node:
            changed |= set_resolution(item, value)
    return changed

if set_resolution(cfg, TARGET_RESOLUTION):
    SPIKE_CONFIG = WORK / "unet_512.yaml"
    SPIKE_CONFIG.write_text(yaml.safe_dump(cfg, sort_keys=False))
    print(f"wrote {SPIKE_CONFIG} with resolution={TARGET_RESOLUTION}")
else:
    SPIKE_CONFIG = UNET_CONFIG
    print("could not locate a resolution field to override; using config as-is")

print(SPIKE_CONFIG.read_text()[:1200])

## 4. Pull the 1.6 weights

From `hf.co/ByteDance/LatentSync-1.6`. This is the slow part — several GB.

Once the spike passes, copy these files into your private Kaggle Dataset. Every
later session then mounts them read-only instead of re-downloading, which is the
single biggest practical speedup in the whole project.

In [ ]:
!pip install --quiet --upgrade huggingface_hub

from huggingface_hub import list_repo_files, snapshot_download

HF_REPO = "ByteDance/LatentSync-1.6"

print(f"files in {HF_REPO}:")
for f in sorted(list_repo_files(HF_REPO)):
    print(f"  {f}")

In [ ]:
CKPT_DIR = REPO_DIR / "checkpoints"
CKPT_DIR.mkdir(exist_ok=True)

# The repo's inference script expects weights under checkpoints/, including the
# Whisper audio encoder and the auxiliary face-detection models.
snapshot_download(repo_id=HF_REPO, local_dir=str(CKPT_DIR), local_dir_use_symlinks=False)

print("\ndownloaded:")
total = 0
for p in sorted(CKPT_DIR.rglob("*")):
    if p.is_file():
        size_mb = p.stat().st_size / 1024**2
        total += size_mb
        if size_mb > 1:
            print(f"  {size_mb:9.1f} MB  {p.relative_to(CKPT_DIR)}")
print(f"  {total:9.1f} MB  TOTAL")

UNET_CKPT = next(CKPT_DIR.rglob("latentsync_unet.pt"), None)
if UNET_CKPT is None:
    candidates = [p for p in CKPT_DIR.rglob("*.pt")] + [p for p in CKPT_DIR.rglob("*.ckpt")]
    raise SystemExit(f"latentsync_unet.pt not found. Candidates: {candidates}")
print(f"\nU-Net checkpoint: {UNET_CKPT.relative_to(REPO_DIR)}")

## 5. Install dependencies

Kaggle already provides torch and ffmpeg, so only the gaps get installed.
Deliberately *not* using the repo's `setup_env.sh` — it pins a torch build that
would fight Kaggle's preinstalled CUDA stack.

In [ ]:
!pip install --quiet diffusers transformers accelerate einops omegaconf \
    decord av mediapipe face-alignment python_speech_features librosa soundfile

# ffmpeg is host-provided on Kaggle -- confirm rather than install.
!ffmpeg -version | head -1
!which ffmpeg ffprobe

## 6. Build a 5-second test clip

Uses your real base loop if the dataset is attached, since that is the footage
the answer actually needs to apply to. Falls back to the repo's own sample.

Five seconds is enough: LatentSync processes video in fixed-size batches, so peak
VRAM is set by batch size and resolution, **not** by total video length. A short
clip measures the same peak a three-minute one would.

In [ ]:
SPIKE_DIR = WORK / "spike"
SPIKE_DIR.mkdir(exist_ok=True)
TEST_VIDEO = SPIKE_DIR / "test_5s.mp4"
TEST_AUDIO = SPIKE_DIR / "test_5s.wav"

asset_video = None
for candidate in Path("/kaggle/input").rglob("base_loop.mp4"):
    asset_video = candidate
    break

if asset_video:
    print(f"using your footage: {asset_video}")
    !ffmpeg -y -loglevel error -i "{asset_video}" -t 5 -an -c:v libx264 -crf 16 -pix_fmt yuv420p "{TEST_VIDEO}"
else:
    sample = next((REPO_DIR / "assets").rglob("*.mp4"), None) if (REPO_DIR / "assets").exists() else None
    if sample:
        print(f"dataset not attached; using repo sample: {sample.name}")
        !ffmpeg -y -loglevel error -i "{sample}" -t 5 -an -c:v libx264 -crf 16 -pix_fmt yuv420p "{TEST_VIDEO}"
    else:
        raise SystemExit(
            "No test footage. Attach the talkinghead-assets dataset via + Add Input.\n"
            "A synthetic clip will not work -- LatentSync needs a real detectable face."
        )

# Any speech works for a VRAM measurement; a tone is enough to drive the model.
asset_audio = next(Path("/kaggle/input").rglob("reference.wav"), None)
if asset_audio:
    !ffmpeg -y -loglevel error -i "{asset_audio}" -t 5 -ac 1 -ar 16000 "{TEST_AUDIO}"
else:
    !ffmpeg -y -loglevel error -f lavfi -i "sine=frequency=200:sample_rate=16000" -t 5 -ac 1 "{TEST_AUDIO}"

!ffprobe -v error -show_entries stream=width,height,duration,codec_type -of default=noprint_wrappers=1 "{TEST_VIDEO}"
!ffprobe -v error -show_entries stream=duration,sample_rate -of default=noprint_wrappers=1 "{TEST_AUDIO}"

## 7. Measure

Two independent measurements, because they catch different things:

- **torch's own counter** (`max_memory_reserved`) — precise, but blind to memory
  allocated outside torch's allocator.
- **an `nvidia-smi` poller** — sees everything the process actually holds,
  including cuDNN workspaces and the CUDA context.

The poller runs in a background thread and records the highest reading it sees.
Inference runs as a subprocess (that is the interface the repo offers), so the
poller is the measurement that matters — torch's counter only sees this process.

In [ ]:
import threading, time

class VramPoller:
    """Samples total GPU memory in use and remembers the peak."""

    def __init__(self, interval=0.25):
        self.interval = interval
        self.peak_mb = 0.0
        self.samples = []
        self._stop = threading.Event()
        self._thread = None

    def _poll(self):
        while not self._stop.is_set():
            try:
                out = subprocess.run(
                    ["nvidia-smi", "--query-gpu=memory.used",
                     "--format=csv,noheader,nounits"],
                    capture_output=True, text=True, timeout=5,
                ).stdout.strip().splitlines()
                used = max(float(v) for v in out if v.strip())
                self.samples.append(used)
                self.peak_mb = max(self.peak_mb, used)
            except Exception:
                pass
            self._stop.wait(self.interval)

    def __enter__(self):
        self._thread = threading.Thread(target=self._poll, daemon=True)
        self._thread.start()
        return self

    def __exit__(self, *exc):
        self._stop.set()
        if self._thread:
            self._thread.join(timeout=5)

    @property
    def peak_gb(self):
        return self.peak_mb / 1024


# Baseline before loading anything, so the model's own cost is separable from
# whatever the session already had resident.
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
baseline = subprocess.run(
    ["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"],
    capture_output=True, text=True,
).stdout.strip().splitlines()
BASELINE_GB = max(float(v) for v in baseline if v.strip()) / 1024
print(f"baseline VRAM in use: {BASELINE_GB:.2f} GB")

In [ ]:
INFERENCE_STEPS = 20   # repo default; more steps costs time, not memory
GUIDANCE_SCALE = 1.5
OUT_VIDEO = SPIKE_DIR / "out_512.mp4"

cmd = [
    sys.executable, "-m", "scripts.inference",
    "--unet_config_path", str(SPIKE_CONFIG),
    "--inference_ckpt_path", str(UNET_CKPT),
    "--video_path", str(TEST_VIDEO),
    "--audio_path", str(TEST_AUDIO),
    "--video_out_path", str(OUT_VIDEO),
    "--inference_steps", str(INFERENCE_STEPS),
    "--guidance_scale", str(GUIDANCE_SCALE),
]
print(" ".join(cmd), "\n")

started = time.time()
with VramPoller() as poller:
    proc = subprocess.run(cmd, cwd=REPO_DIR, capture_output=True, text=True)
elapsed = time.time() - started

PEAK_GB = poller.peak_gb
MODEL_GB = max(0.0, PEAK_GB - BASELINE_GB)
OOM = "out of memory" in (proc.stdout + proc.stderr).lower()
SUCCESS = proc.returncode == 0 and OUT_VIDEO.exists()

print(f"exit code      {proc.returncode}")
print(f"elapsed        {elapsed:.0f}s for 5s of video")
print(f"peak VRAM      {PEAK_GB:.2f} GB  (model contribution ~{MODEL_GB:.2f} GB)")
print(f"samples taken  {len(poller.samples)}")
print(f"OOM reported   {OOM}")
print(f"output written  {OUT_VIDEO.exists()}")

if not SUCCESS:
    print("\n--- stderr (last 40 lines) ---")
    print("\n".join(proc.stderr.strip().splitlines()[-40:]))
    print("\n--- stdout (last 20 lines) ---")
    print("\n".join(proc.stdout.strip().splitlines()[-20:]))

## 8. Verdict

A margin is reserved rather than accepting "it fit once." Real renders run longer
than five seconds and Kaggle sessions have other things resident, so a result
that only just fits will fail intermittently — which is worse than choosing the
720p profile deliberately.

In [ ]:
SAFETY_MARGIN_GB = 1.5
usable = TOTAL_VRAM_GB - SAFETY_MARGIN_GB

print("=" * 68)
print("PHASE 0 VERDICT")
print("=" * 68)
print(f"  GPU              {props.name}")
print(f"  total VRAM       {TOTAL_VRAM_GB:.2f} GB")
print(f"  usable budget    {usable:.2f} GB  (after {SAFETY_MARGIN_GB} GB margin)")
print(f"  peak observed    {PEAK_GB:.2f} GB")
print(f"  render speed     {elapsed / 5:.1f}s of compute per 1s of video")
print()

if SUCCESS and PEAK_GB <= usable:
    headroom = usable - PEAK_GB
    print(f"  RESULT: 512 FITS, with {headroom:.2f} GB to spare.")
    print()
    print("  -> Use the 1080p profile (the default). No config change needed.")
    print(f"  -> Estimated GPU time for a 2-minute video: "
          f"{elapsed / 5 * 120 / 60:.0f} minutes")
    print(f"  -> Freeze this commit in config.TRUSTED_SOURCES:")
    print(f"       latentsync revision = {LATENTSYNC_SHA}")
    VERDICT = "1080p"
elif SUCCESS:
    print(f"  RESULT: 512 ran, but peaked at {PEAK_GB:.2f} GB against a "
          f"{usable:.2f} GB budget.")
    print()
    print("  -> Too tight to rely on. Longer renders will OOM intermittently,")
    print("     which is harder to debug than choosing 720p up front.")
    print("  -> Set TH_PROFILE=720p, or retry with a smaller batch size in the")
    print("     U-Net config and re-run this notebook.")
    VERDICT = "720p"
else:
    print("  RESULT: inference FAILED." + (" Cause: out of memory." if OOM else ""))
    print()
    if OOM:
        print("  -> 512 does not fit on this GPU. Set TH_PROFILE=720p.")
        print("     This is the documented fallback and produces good output;")
        print("     downscaling 1080p source to 720p hides most of the softness")
        print("     from the smaller 256 face crop.")
        VERDICT = "720p"
    else:
        print("  -> Failed for a reason other than memory. Read the stderr above")
        print("     before concluding anything about VRAM -- a missing dependency")
        print("     or an undetected face looks nothing like an OOM.")
        VERDICT = "inconclusive"

print()
print(f"  VERDICT: {VERDICT}")
print("=" * 68)

## 9. Look at the output

The number answers whether it *runs*. Only your eyes answer whether it is good.

Watch the mouth specifically: do the teeth read as teeth at full resolution, or
as a smear? That is the exact failure 1.6 was trained to fix, so this is where
you confirm it actually did.

In [ ]:
from IPython.display import Video, display

if OUT_VIDEO.exists():
    !ffprobe -v error -show_entries stream=width,height,nb_frames -of default=noprint_wrappers=1 "{OUT_VIDEO}"
    # Crop to the face region and blow it up, so mouth detail is actually
    # visible rather than lost in a small inline player.
    ZOOM = SPIKE_DIR / "out_512_mouth.mp4"
    !ffmpeg -y -loglevel error -i "{OUT_VIDEO}" -vf "crop=iw/3:ih/3:iw/3:ih/2,scale=640:-2" -an "{ZOOM}"
    print("\nfull frame:")
    display(Video(str(OUT_VIDEO), embed=True, width=720))
    print("mouth region, zoomed:")
    display(Video(str(ZOOM), embed=True, width=640))
else:
    print("No output video -- see the failure output above.")

## What to do next

**If the verdict was `1080p`:** nothing to change — that is the default profile.
Freeze the printed commit SHA into `config.TRUSTED_SOURCES`, copy
`checkpoints/` into your private Kaggle Dataset so future sessions skip the
download, and move on to Phase 2 (the Chatterbox voice provider).

**If the verdict was `720p`:** set `TH_PROFILE=720p` and carry on. This was the
planned fallback, not a failure — you still get a good video, and the 1080p
source downscaling to 720p is what conceals the smaller face crop.

**If it was `inconclusive`:** the failure was not about memory. Read the stderr
in cell 7 before changing anything.

Either way, save this notebook's output — it is the evidence behind the profile
choice, and worth having when someone asks why.